In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
import numpy as np
import torch
from torch import nn
import json
from collections import Counter

Загрузим датасет

In [4]:
tags_to_names = {
    "cs": "Computer Science",
    "econ": "Economics",
    "eess": "Electrical Engineering and Systems Science",
    "math": "Mathematics",
    "physics": "Physics",
    "q-bio": "Quantitive Biology",
    "q-fin": "Quantitive Finance",
    "stat": "Statistics"
}

In [ ]:
from huggingface_hub import login

login("")

In [5]:
ds_balanced = load_dataset("pinmax/arxiv_balanced_soft_labels")

Скачаем уже предобученную модель distilbert с HF

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("distilbert/distilbert-base-uncased", num_labels=len(tags_to_names))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Токенизируем abstract и title в датасете

In [7]:
def tokenize_function(examples):
    return tokenizer(examples["title"], examples["abstract"], padding="max_length", truncation=True, max_length=512)

In [8]:
ds_balanced = ds_balanced.map(tokenize_function, keep_in_memory=True)

Map:   0%|          | 0/139511 [00:00<?, ? examples/s]

Map:   0%|          | 0/34878 [00:00<?, ? examples/s]

In [14]:
ds_balanced

DatasetDict({
    train: Dataset({
        features: ['title', 'abstract', 'label', 'soft_labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 139511
    })
    test: Dataset({
        features: ['title', 'abstract', 'label', 'soft_labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 34878
    })
})

In [15]:
ds_balanced.set_format("torch", columns=["input_ids", "attention_mask", "token_type_ids", "soft_labels"])

In [16]:
tokenizer.decode(ds_balanced['test'][0]['input_ids'])

'[CLS] combinatorial rna design : designability and structure - approximating algorithm [SEP] in this work, we consider the combinatorial rna design problem, a minimal instance of the rna design problem which aims at finding a sequence that admits a given target as its unique base pair maximizing structure. we provide complete characterizations for the structures that can be designed using restricted alphabets. under a classic four - letter alphabet, we provide a complete characterization of designable structures without unpaired bases. when unpaired bases are allowed, we provide partial characterizations for classes of designable / undesignable structures, and show that the class of designable structures is closed under the stutter operation. membership of a given structure to any of the classes can be tested in linear time and, for positive instances, a solution can be found in linear time. finally, we consider a structure - approximating version of the problem that allows to extend 

In [ ]:
TrainingArguments?

Дообучим модель на получившемся датасете

In [ ]:
login("")

In [25]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    true_labels = np.argmax(labels, axis=-1)

    return {
        "soft_labels_accuracy": float(np.mean(labels[np.arange(len(preds)), preds])),
        "f1": f1_score(true_labels, preds, average="weighted"),
        "coverage": float(np.mean([labels[i, preds[i]] > 0 for i in range(len(preds))])),
    }

args = TrainingArguments(
    output_dir="./article-classifier-scibert",
    num_train_epochs=5,
    learning_rate=3e-5,
    lr_scheduler_type="cosine",

    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    fp16=True,

    warmup_steps=300,

    logging_strategy="epoch",
    logging_steps=1,
    save_strategy="epoch",
    save_total_limit=2,
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="coverage",
    greater_is_better=True,
    hub_model_id="pinmax/article-classifier-distilbert-soft-labels",
    push_to_hub=True,

    disable_tqdm=False,
    dataloader_num_workers=2,

    optim="adamw_torch",
    weight_decay=0.05,
    use_cpu=False,
)



In [26]:
class SoftLabelTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        soft_labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        logits = outputs.logits

        log_probs = nn.functional.log_softmax(logits, dim=-1)
        loss = nn.functional.kl_div(log_probs, soft_labels.float(), reduction="batchmean")

        return (loss, outputs) if return_outputs else loss

In [ ]:
def collate_fn(batch):
    input_ids = torch.stack([x["input_ids"] for x in batch])
    attention_mask = torch.stack([x["attention_mask"] for x in batch])
    labels = torch.stack([x["soft_labels"] for x in batch])
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

trainer = SoftLabelTrainer(
    model=model,
    args=args,
    train_dataset=ds_balanced["train"],
    eval_dataset=ds_balanced["test"],
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Soft Labels Accuracy,F1,Coverage
1,0.493507,0.388176,0.729148,0.694185,0.921182
2,0.347785,0.358639,0.735597,0.725701,0.928608
3,0.283859,0.361251,0.737288,0.705542,0.930013


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Посчитаем метрики по основному label

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("pinmax/article-classifier-distilbert-soft-labels", num_labels=len(tags_to_names))

In [18]:
ds_balanced.reset_format()

In [23]:
true_labels = list(ds_balanced['test']['label'])

In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [26]:
from tqdm import tqdm


batch_size = 32
all_preds = []
model.to(device)
for i in tqdm(range(0, len(ds_balanced['test']), batch_size)):
    batch = ds_balanced['test'].select(range(i, min(i + batch_size, len(ds_balanced['test']))))
    
    input_ids = torch.tensor(np.array(batch['input_ids'])).to(device)
    attention_mask = torch.tensor(np.array(batch['attention_mask'])).to(device)
    token_type_ids = torch.tensor(np.array(batch['token_type_ids'])).to(device)
    
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
    
    preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
    all_preds.extend(preds.tolist())


100%|██████████| 1090/1090 [11:23<00:00,  1.59it/s]


In [29]:
print("Accuracy: ", accuracy_score(true_labels, all_preds))
print("F1:       ", f1_score(true_labels, all_preds, average="weighted"))
print("Precision:", precision_score(true_labels, all_preds, average="weighted", zero_division=0))
print("Recall:   ", recall_score(true_labels, all_preds, average="weighted", zero_division=0))

Accuracy:  0.8416193589081943
F1:        0.8432501892276004
Precision: 0.8600285768514431
Recall:    0.8416193589081943
